# MDLM (Masked Diffusion Language Model) — HuggingFace 模型调用示例

**论文**: [Simple and Effective Masked Diffusion Language Models](http://arxiv.org/abs/2406.07524) (Sahoo et al., 2024, NeurIPS)

**仓库**: https://github.com/kuleshov-group/mdlm

---

### 本 Notebook 功能

1. 加载 HuggingFace 上的预训练 MDLM checkpoint
2. 从全 `[MASK]` 状态开始逐步去噪生成文本
3. **可视化去噪过程**：展示词语逐步浮现的中间状态
4. 半自回归 (Semi-AR) 扩展生成超长文本

### 模型信息

| 属性 | 值 |
|------|-----|
| 模型 | `kuleshov-group/mdlm-no_flashattn-fp32-owt` |
| 参数量 | ~1.3 亿（类似 GPT2-medium） |
| 上下文长度 | 1024 token |
| 训练数据 | OpenWebText (~33B tokens) |
| 精度 | fp32（T4 兼容，无需 flash attention） |
| 采样步数 | 1000 步 |

---

## 快速开始

将 `mdlm_chinese/` 整个文件夹上传到 Colab，打开本 notebook，依次运行即可。

---
## 1. 安装依赖

> ⚠️ 如遇 NumPy 版本冲突，运行后重启运行时再执行后续 cell。

In [ ]:
# ============================================================
# 安装 MDLM 运行所需的所有依赖
# 注意：本仓库已修改为无需 flash_attn（使用原生 attention）
# ============================================================

# PyTorch（CUDA 12.1，Colab T4 兼容）
!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu121

# HuggingFace 相关
!pip install transformers==4.38.2 datasets einops==0.7.0

# 训练框架
!pip install lightning==2.2.1 hydra-core==1.3.2 omegaconf==2.3.0

# 工具库
!pip install fsspec h5py packaging==23.2 pandas rich==13.7.1 seaborn==0.13.2 scikit-learn==1.4.0 wandb==0.13.5

# 确保 NumPy 版本正确
!pip install numpy==1.26.4

---
## 2. 设置工作目录

上传 `mdlm_chinese/` 文件夹后，将其设置为工作目录。

In [ ]:
# ============================================================
# 上传代码：将 mdlm_chinese.zip 上传到 Colab，本 cell 自动解压
# ============================================================
import os, zipfile

ZIP_PATH = '/content/mdlm_chinese.zip'
EXTRACT_DIR = '/content/mdlm_chinese'

if os.path.exists(ZIP_PATH):
    print(f'解压 {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    os.chdir(EXTRACT_DIR)
elif os.path.exists(EXTRACT_DIR):
    os.chdir(EXTRACT_DIR)
else:
    print('请上传 mdlm_chinese.zip (在 Colab 左侧文件图标 → 上传)')
print(f'工作目录: {os.getcwd()}')

---
## 3. 导入模块并检查 NumPy 版本

MDLM 依赖 NumPy 1.26.4，如果版本不对需要重启运行时。

In [ ]:
# ============================================================
# 导入 MDLM 模块（diffusion.py, main.py, dataloader.py 等）
# 检查 NumPy 版本，确保与依赖兼容
# ============================================================
import numpy as np
print(f'NumPy 版本: {np.__version__}')

if np.__version__ != '1.26.4':
    # 版本不匹配时需要重启运行时
    print('NumPy 版本不匹配！请重启运行时：运行时 → 重新启动并全部运行')
    print('重启后不用再运行 pip install cell，直接从此 cell 开始即可')
    os.kill(os.getpid(), 9)

# 导入 MDLM 模块
import fsspec
import hydra
import lightning as L
import omegaconf
import rich.syntax
import rich.tree
import torch

import dataloader
import diffusion
import main
import utils

print('所有模块导入成功！')

---
## 4. 配置参数

使用 Hydra 加载配置文件，并覆盖为采样模式。

**关键参数说明**:

| 参数 | 值 | 说明 |
|------|-----|------|
| `mode` | `sample_eval` | 采样评估模式 |
| `backbone` | `hf_dit` | HuggingFace DiT 主干 |
| `sampling.predictor` | `ddpm_cache` | 带缓存的 DDPM 采样器 |
| `sampling.steps` | `1000` | 采样步数 |
| `sampling.visualize` | `True` | **启用快照可视化** |
| `model.length` | `1024` | 序列长度 |
| `loader.eval_batch_size` | `1` | 推理批大小 |

In [ ]:
# ============================================================
# 配置 Hydra 参数覆盖
#
# 这里使用 Hydra 的 compose API 来动态配置覆盖，
# 相当于在命令行执行：
#   python main.py mode=sample_eval ... visualize=True
# ============================================================

from omegaconf import DictConfig

overrides=[
    'mode=sample_eval',                            # 采样评估模式
    'eval.checkpoint_path=kuleshov-group/mdlm-no_flashattn-fp32-owt',  # HF 模型名
    'data=openwebtext-split',                      # 数据集（用于 tokenizer）
    'model.length=1024',                           # 序列长度 1024
    'sampling.predictor=ddpm_cache',               # 采样器
    'sampling.steps=1000',                         # 采样步数
    'loader.eval_batch_size=1',                    # 批大小 1
    'sampling.num_sample_batches=1',               # 只生成 1 批
    'backbone=hf_dit',                             # HuggingFace 主干
    '+sampling.visualize=True',                    # 启用快照可视化
]

with hydra.initialize(version_base=None, config_path='configs'):
    config = hydra.compose(
        config_name='config',
        overrides=overrides
    )

print('配置加载完成')
print(f'采样步数: {config.sampling.steps}')
print(f'采样器: {config.sampling.predictor}')
print(f'可视化: {config.sampling.visualize}')

---
## 5. 生成文本 + 快照可视化

这是最核心的一步：

1. 从 HuggingFace Hub 加载预训练模型
2. 从全 `[MASK]` 状态开始迭代去噪
3. **在 t=0.5, 0.4, ..., 0.0 处记录中间状态快照**
4. 打印渐进式去噪过程，并保存到 `snapshots.txt`

> ⏱ T4 GPU 上 1000 步约需 2-5 分钟

In [ ]:
# ============================================================
# 加载模型 + 采样生成 + 快照可视化
#
# 流程:
#   1. L.seed_everything() 固定随机种子
#   2. dataloader.get_tokenizer() 获取 GPT-2 tokenizer
#   3. main.generate_samples() 自动处理 visualize=True 分支
#      - 调用 Diffusion.restore_model_and_sample(ret_snapshots=True)
#      - 调用 Diffusion.display_snapshots() 打印可视化
#      - 调用 Diffusion.save_snapshots_to_file() 保存到文件
# ============================================================

L.seed_everything(config.seed, workers=True)
print(f'随机种子: {config.seed}')

# 获取 tokenizer（GPT-2 tokenizer，与 HF 模型兼容）
tokenizer = dataloader.get_tokenizer(config)
print(f'词表大小: {tokenizer.vocab_size}')

# 生成样本（包含快照可视化）
logger = utils.get_logger(__name__)
samples = main.generate_samples(config, logger, tokenizer)

print('\n' + '=' * 70)
print('最终生成文本:')
for i, s in enumerate(samples):
    print(f'样本 {i+1}: {s}')

---
## 6. 查看保存的快照文件

可视化结果已保存到 `snapshots.txt`，可以用 cat 查看完整内容。

In [ ]:
# 查看快照文件的前 50 行
!head -50 snapshots.txt

---
## 7. 半自回归 (Semi-AR) 扩展生成

MDLM 支持通过 stride 机制生成长度超过模型最大长度（1024）的文本。

原理：每次生成 stride_length 个新 token，拼接到已有上下文后作为下一步的条件。

> ⚠️ 注意：semi_ar 模式目前不支持快照可视化。

In [ ]:
# ============================================================
# 半自回归采样：生成 2048 个 token 的长文本
#
# 参数:
#   semi_ar = True       启用半自回归模式
#   stride_length = 512  每次扩展 512 个 token
#   num_strides = 2      扩展 2 次
#   总长度 = 1024 + 2*512 = 2048
# ============================================================

sar_config = config.copy()
sar_config['sampling']['semi_ar'] = True
sar_config['sampling']['stride_length'] = 512
sar_config['sampling']['num_strides'] = 2

sar_samples = main.generate_samples(sar_config, logger, tokenizer)

print('\n' + '=' * 70)
print('半自回归生成结果 (长度 2048):')
for i, s in enumerate(sar_samples):
    print(f'样本 {i+1}: {s[:500]}...')  # 只显示前 500 字符

---
## 常见问题

### Q: 生成质量不好怎么办？
- 增加 `sampling.steps`（如 2000 步）
- 更换随机种子 `config.seed`
- 使用 `ddpm` 采样器（更慢但更准）替代 `ddpm_cache`

### Q: 如何关闭可视化？
- 在 overrides 中删除 `+sampling.visualize=True` 或将值改为 `False`

### Q: CUDA 显存不足？
- 确保使用 `mdlm-no_flashattn-fp32-owt` 模型（fp32 版）
- 减小 `model.length` 到 512
- T4 (16GB) 可以正常运行 1024 长度

### Q: 快照文件在哪？
- 保存在 `snapshots.txt`，包含从 t=1.0 到 t=0.0 的完整去噪过程
- `[MASK]` 表示该位置 token 尚未被确定

### Q: 可视化感觉「跳跃」不连续？
- 这是 MDLM 的正常行为：大部分去噪发生在最后 20% 的时间步
- 快照 t 值集中在低 t 区域（0.5→0.0），已针对此特点优化

In [ ]:
# 清理临时文件
import glob
for f in glob.glob('*.txt'):
    if f != 'snapshots.txt':
        os.remove(f)
        print(f'已删除: {f}')
print('清理完成')